In [1]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# =========================
# 2. LOAD DATASET
# =========================
# Change path to your downloaded CSV
df = pd.read_csv("housing.csv")

print("Dataset shape:", df.shape)
print(df.head())

# =========================
# 3. DEFINE FEATURES & TARGET
# =========================
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

# =========================
# 4. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 5. COLUMN TYPES
# =========================
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

# =========================
# 6. PREPROCESSING PIPELINES
# =========================

# Numerical pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# =========================
# 7. FULL PIPELINE (PREPROCESS + MODEL)
# =========================
model = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

# =========================
# 8. TRAIN MODEL
# =========================
model.fit(X_train, y_train)

# =========================
# 9. PREDICTION
# =========================
y_pred = model.predict(X_test)

# =========================
# 10. EVALUATION
# =========================
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Performance:")
print("RMSE:", rmse)
print("R2 Score:", r2)

# =========================
# 11. SAMPLE PREDICTION
# =========================
sample = X_test.iloc[:5]
predictions = model.predict(sample)

print("\nSample Predictions:")
for i in range(len(sample)):
    print(f"Predicted: {predictions[i]:.2f}, Actual: {y_test.iloc[i]}")

Dataset shape: (20640, 10)
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


C:\Users\sharm\AppData\Local\Temp\ipykernel_2224\4182815230.py:41: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns



Model Performance:
RMSE: 49316.15165472101
R2 Score: 0.8144027662801667

Sample Predictions:
Predicted: 54278.96, Actual: 47700.0
Predicted: 69582.46, Actual: 45800.0
Predicted: 466598.43, Actual: 500001.0
Predicted: 251144.45, Actual: 218600.0
Predicted: 263479.47, Actual: 278000.0
